In [ ]:
import grewpy
from grewpy import Corpus, CorpusDraft, Request
from collections import Counter
import pandas as pd

import sys
sys.path.insert(1, "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/tod")

import tod.corpus
import tod.outliers
import tod.clustering
import tod.plotting
import tod.dimension_reduction_classic

GREW_PATTERN = "pattern{X[upos<>PUNCT]}"
PATTERNS_TEXT_FILE = "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/scripts/3. probability_matrix/patterns_all_nodes.txt"




    

In [ ]:
def get_inventory_matrix(corpus):
    NotImplementedError("This function should be implemented to extract the inventory matrix from the corpus.")

In [ ]:
def get_transition_matrix(corpus):
    NotImplementedError("This function should be implemented to extract the transition matrix from the corpus.")

In [ ]:
def get_variance_matrix(corpus):
    NotImplementedError("This function should be implemented to extract the variance matrix from the corpus.")

In [ ]:
def process_treebank(path, language_name):

    corpus = tod.corpus.Corpus(
            treebank_path=path,
            grew_pattern=GREW_PATTERN,
            patterns_text_file=PATTERNS_TEXT_FILE,
            use_sud=False,
            matrix_type="coverage",
            excluded_feature_patterns=[r"own"],
            included_feature_patterns=[r"upos=", 
                                        r"position=", 
                                        r"rel_shallow=",
                                        r"Abbr=", 
                                        r"Aspect=",  
                                        r"Animacy=",  
                                        r"Case=",  
                                        r"Clusivity=",  
                                        r"Definite=",  
                                        r"Deixis=", 
                                        r"DeixisRef",  
                                        r"Evident=",  
                                        r"Negation=",
                                        r"Number=",  
                                        r"Gender=",  
                                        r"Degree=",  
                                        r"ExtPos=", 
                                        r"Foreign=", 
                                        r"Mood",  
                                        r"NounClass=",  
                                        r"NumType=",  
                                        r"Person=",  
                                        r"Polarity=", 
                                        r"Polite=",  
                                        r"Poss=",  
                                        r"PronType=",  
                                        r"Reflex=",  
                                        r"Tense=",  
                                        r"Typo=",     
                                        r"VerbForm=",  
                                        r"Voice=",   ])
    # feature_matrix = corpus.feature_matrix
    inventory_matrix = get_inventory_matrix(corpus)
    transition_matrix = get_transition_matrix(corpus)
    variance_matrix = get_variance_matrix(corpus)
    master_matrix_row = pd.concat([inventory_matrix, transition_matrix, variance_matrix], axis=1)
    return master_matrix_row


In [ ]:
def select_features(master_matrix, main_category, variance_categories):
    NotImplementedError("This function should be implemented to select the relevant features from the master matrix based on the main category and variance categories.")

In [ ]:
from tqdm.auto import tqdm
import os
# treebanks_folder = "/Users/madalina/Documents/PHD/code/data/small_test"
treebanks_folder = "/Users/madalina/Documents/PHD/code/data/single_treebank"
treebanks = [t for t in os.listdir(treebanks_folder) if t.startswith("UD_")]

all_language_data = []
for treebank in tqdm(treebanks, desc="Processing treebanks"):
    print(f"Processing {treebank}...")
    full_path = os.path.join(treebanks_folder, treebank)
    try:
        lang_name_list = treebank.split("_")[1:]
        new_s = "_".join(lang_name_list)
        master_matrix_row = process_treebank(full_path, new_s)
        all_language_data.append(master_matrix_row)
    except Exception as e:
        print(f"Failed {treebank}: {e}")

master_df = pd.DataFrame(all_language_data).set_index("Language")

In [ ]:
import gower

main_comparison_category = "ADJ"
variance_categories = ["NOUN", "VERB"]

pos_df = select_features(master_df, main_comparison_category, variance_categories)
distance_matrix = gower.gower_matrix(pos_df)